# Olist E-Commerce Data Cleaning Pipeline

This notebook transforms the raw Olist datasets into validated, analysis-ready tables while preserving the original source files.

## Cleaning Principles

- Never modify raw source files
- Preserve legitimate incomplete records
- Remove only verified duplicates
- Document all manual mappings
- Validate record counts and relationships after transformation

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))

raw_datasets = {
    file.stem: pd.read_csv(file, low_memory=False)
    for file in csv_files
}

print(f"Loaded {len(raw_datasets)} raw datasets.")

for name, df in raw_datasets.items():
    print(f"{name}: {len(df):,} rows")

Project root: /Users/adnan/Documents/ecommerce-product-intelligence
Raw data directory: /Users/adnan/Documents/ecommerce-product-intelligence/data/raw
Loaded 9 raw datasets.
olist_customers_dataset: 99,441 rows
olist_geolocation_dataset: 1,000,163 rows
olist_order_items_dataset: 112,650 rows
olist_order_payments_dataset: 103,886 rows
olist_order_reviews_dataset: 99,224 rows
olist_orders_dataset: 99,441 rows
olist_products_dataset: 32,951 rows
olist_sellers_dataset: 3,095 rows
product_category_name_translation: 71 rows


## 1. Standardize Date and Status Fields

Timestamp columns are converted from text into datetime values so delivery speed, delays, trends, and order lifecycle metrics can be calculated accurately.

In [4]:
customers = raw_datasets[
    "olist_customers_dataset"
].copy()

geolocation = raw_datasets[
    "olist_geolocation_dataset"
].copy()

order_items = raw_datasets[
    "olist_order_items_dataset"
].copy()

payments = raw_datasets[
    "olist_order_payments_dataset"
].copy()

reviews = raw_datasets[
    "olist_order_reviews_dataset"
].copy()

orders = raw_datasets[
    "olist_orders_dataset"
].copy()

products = raw_datasets[
    "olist_products_dataset"
].copy()

sellers = raw_datasets[
    "olist_sellers_dataset"
].copy()

translations = raw_datasets[
    "product_category_name_translation"
].copy()

print("Working copies created successfully.")

Working copies created successfully.


In [5]:
date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

working_tables = {
    "orders": orders,
    "order_items": order_items,
    "reviews": reviews
}

date_conversion_results = []

for table_name, columns in date_columns.items():
    df = working_tables[table_name]

    for column in columns:
        missing_before = int(df[column].isna().sum())

        df[column] = pd.to_datetime(
            df[column],
            errors="coerce"
        )

        missing_after = int(df[column].isna().sum())

        date_conversion_results.append({
            "table": table_name,
            "column": column,
            "missing_before": missing_before,
            "missing_after": missing_after,
            "invalid_values_created": (
                missing_after - missing_before
            ),
            "new_data_type": str(df[column].dtype)
        })

date_conversion_audit = pd.DataFrame(
    date_conversion_results
)

date_conversion_audit

,table,column,missing_before,missing_after,invalid_values_created,new_data_type
0,orders,order_purchase_timestamp,0,0,0,datetime64[us]
1,orders,order_approved_at,160,160,0,datetime64[us]
2,orders,order_delivered_carrier_date,1783,1783,0,datetime64[us]
3,orders,order_delivered_customer_date,2965,2965,0,datetime64[us]
4,orders,order_estimated_delivery_date,0,0,0,datetime64[us]
5,order_items,shipping_limit_date,0,0,0,datetime64[us]
6,reviews,review_creation_date,0,0,0,datetime64[us]
7,reviews,review_answer_timestamp,0,0,0,datetime64[us]


## 2. Clean Product Categories

Missing product categories are labeled as `unknown`. Two legitimate Portuguese categories missing from the supplied reference table are added through documented manual mappings.

In [6]:
products_before = len(products)

products["category_was_missing"] = (
    products["product_category_name"].isna()
)

products["product_category_name"] = (
    products["product_category_name"]
    .fillna("unknown")
    .str.strip()
    .str.lower()
)

manual_translations = pd.DataFrame({
    "product_category_name": [
        "pc_gamer",
        "portateis_cozinha_e_preparadores_de_alimentos",
        "unknown"
    ],
    "product_category_name_english": [
        "gaming_pc",
        "portable_kitchen_and_food_preparation_appliances",
        "unknown"
    ]
})

translations_clean = (
    pd.concat(
        [translations, manual_translations],
        ignore_index=True
    )
    .drop_duplicates(
        subset=["product_category_name"],
        keep="first"
    )
)

products = products.merge(
    translations_clean,
    how="left",
    on="product_category_name",
    validate="many_to_one"
)

products = products.rename(
    columns={
        "product_category_name_english":
        "product_category"
    }
)

assert len(products) == products_before
assert products["product_category"].isna().sum() == 0

print(f"Products before cleaning: {products_before:,}")
print(f"Products after cleaning:  {len(products):,}")
print(
    "Products originally missing a category:",
    f"{products['category_was_missing'].sum():,}"
)
print(
    "Missing English categories after mapping:",
    products["product_category"].isna().sum()
)

Products before cleaning: 32,951
Products after cleaning:  32,951
Products originally missing a category: 610
Missing English categories after mapping: 0


## 3. Clean Geolocation Data

Exact duplicate coordinates are removed. Clearly invalid coordinates outside broad Brazilian geographic boundaries are excluded, and the median latitude and longitude are calculated for each ZIP-code prefix. The median limits the influence of geographic outliers.

In [7]:
geolocation_before = len(geolocation)

geolocation_deduplicated = (
    geolocation
    .drop_duplicates()
    .copy()
)

duplicate_rows_removed = (
    geolocation_before - len(geolocation_deduplicated)
)

valid_coordinate_mask = (
    geolocation_deduplicated["geolocation_lat"]
    .between(-35, 6)
    &
    geolocation_deduplicated["geolocation_lng"]
    .between(-75, -32)
)

invalid_coordinates_removed = int(
    (~valid_coordinate_mask).sum()
)

geolocation_valid = geolocation_deduplicated[
    valid_coordinate_mask
].copy()

geolocation_clean = (
    geolocation_valid
    .groupby(
        "geolocation_zip_code_prefix",
        as_index=False
    )
    .agg(
        latitude=("geolocation_lat", "median"),
        longitude=("geolocation_lng", "median")
    )
)

assert geolocation_clean[
    "geolocation_zip_code_prefix"
].is_unique

print(f"Original rows:             {geolocation_before:,}")
print(
    f"Exact duplicates removed:  "
    f"{duplicate_rows_removed:,}"
)
print(
    f"Invalid coordinates removed: "
    f"{invalid_coordinates_removed:,}"
)
print(
    f"Unique ZIP prefixes:       "
    f"{len(geolocation_clean):,}"
)

Original rows:             1,000,163
Exact duplicates removed:  261,831
Invalid coordinates removed: 25
Unique ZIP prefixes:       19,011


## 4. Standardize and Enrich Locations

Customer and seller city/state fields are standardized, then enriched with median ZIP-prefix coordinates from the cleaned geolocation reference table.

In [8]:
customers_before = len(customers)
sellers_before = len(sellers)

customers["customer_city"] = (
    customers["customer_city"]
    .str.strip()
    .str.lower()
)

customers["customer_state"] = (
    customers["customer_state"]
    .str.strip()
    .str.upper()
)

sellers["seller_city"] = (
    sellers["seller_city"]
    .str.strip()
    .str.lower()
)

sellers["seller_state"] = (
    sellers["seller_state"]
    .str.strip()
    .str.upper()
)

customer_geolocation = geolocation_clean.rename(
    columns={
        "geolocation_zip_code_prefix":
            "customer_zip_code_prefix",
        "latitude": "customer_latitude",
        "longitude": "customer_longitude"
    }
)

seller_geolocation = geolocation_clean.rename(
    columns={
        "geolocation_zip_code_prefix":
            "seller_zip_code_prefix",
        "latitude": "seller_latitude",
        "longitude": "seller_longitude"
    }
)

customers = customers.merge(
    customer_geolocation,
    how="left",
    on="customer_zip_code_prefix",
    validate="many_to_one"
)

sellers = sellers.merge(
    seller_geolocation,
    how="left",
    on="seller_zip_code_prefix",
    validate="many_to_one"
)

assert len(customers) == customers_before
assert len(sellers) == sellers_before

customers["customer_location_available"] = (
    customers["customer_latitude"].notna()
)

sellers["seller_location_available"] = (
    sellers["seller_latitude"].notna()
)

customer_coverage = (
    customers["customer_location_available"].mean() * 100
)

seller_coverage = (
    sellers["seller_location_available"].mean() * 100
)

print(f"Customer location coverage: {customer_coverage:.2f}%")
print(f"Seller location coverage:   {seller_coverage:.2f}%")
print("Customer and seller row counts preserved.")

Customer location coverage: 99.72%
Seller location coverage:   99.77%
Customer and seller row counts preserved.


## 5. Engineer Order-Performance Metrics

New analytical fields measure approval speed, actual delivery time, estimated delivery time, and whether completed orders arrived after their promised date. Missing delivery outcomes remain unknown rather than being incorrectly labeled as on time.

In [9]:
orders["purchase_date"] = (
    orders["order_purchase_timestamp"].dt.date
)

orders["purchase_month"] = (
    orders["order_purchase_timestamp"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

orders["approval_hours"] = (
    orders["order_approved_at"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / 3600

orders["actual_delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

orders["estimated_delivery_days"] = (
    orders["order_estimated_delivery_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

orders["days_from_estimate"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

valid_delivery_comparison = (
    orders["order_delivered_customer_date"].notna()
    & orders["order_estimated_delivery_date"].notna()
)

orders["is_late_delivery"] = pd.Series(
    pd.NA,
    index=orders.index,
    dtype="boolean"
)

orders.loc[
    valid_delivery_comparison,
    "is_late_delivery"
] = (
    orders.loc[
        valid_delivery_comparison,
        "order_delivered_customer_date"
    ]
    >
    orders.loc[
        valid_delivery_comparison,
        "order_estimated_delivery_date"
    ]
)

orders["timeline_anomaly"] = (
    (orders["approval_hours"] < 0)
    | (orders["actual_delivery_days"] < 0)
)

print("Engineered metric summary:")
display(
    orders[
        [
            "approval_hours",
            "actual_delivery_days",
            "estimated_delivery_days",
            "days_from_estimate"
        ]
    ]
    .describe()
    .round(2)
)

evaluated_deliveries = int(
    orders["is_late_delivery"].notna().sum()
)

late_deliveries = int(
    orders["is_late_delivery"].fillna(False).sum()
)

late_delivery_rate = (
    late_deliveries / evaluated_deliveries * 100
)

print(f"Evaluated deliveries: {evaluated_deliveries:,}")
print(f"Late deliveries:      {late_deliveries:,}")
print(f"Late-delivery rate:   {late_delivery_rate:.2f}%")
print(
    "Timeline anomalies:",
    f"{orders['timeline_anomaly'].sum():,}"
)

Engineered metric summary:


,approval_hours,actual_delivery_days,estimated_delivery_days,days_from_estimate
count,99281.00,96476.00,99441.00,96476.00
mean,10.42,12.56,23.77,-11.18
std,26.04,9.55,8.83,10.19
min,0.00,0.53,1.65,-146.02
25%,0.22,6.77,18.33,-16.24
50%,0.34,10.22,23.24,-11.95
75%,14.58,15.72,28.42,-6.39
max,4509.18,209.63,155.14,188.98


Evaluated deliveries: 96,476
Late deliveries:      7,827
Late-delivery rate:   8.11%
Timeline anomalies: 0


### Operational Outlier Treatment

Valid but unusually long approval and delivery times are retained because they may represent genuine operational failures. Values above the 99th percentile are flagged for separate analysis instead of being deleted.

In [10]:
approval_threshold = orders[
    "approval_hours"
].quantile(0.99)

delivery_threshold = orders[
    "actual_delivery_days"
].quantile(0.99)

orders["approval_time_outlier"] = (
    orders["approval_hours"] > approval_threshold
)

orders["delivery_time_outlier"] = (
    orders["actual_delivery_days"] > delivery_threshold
)

print(
    f"99th-percentile approval threshold: "
    f"{approval_threshold:.2f} hours"
)

print(
    f"99th-percentile delivery threshold: "
    f"{delivery_threshold:.2f} days"
)

print(
    "Approval-time outliers:",
    f"{orders['approval_time_outlier'].sum():,}"
)

print(
    "Delivery-time outliers:",
    f"{orders['delivery_time_outlier'].sum():,}"
)

outliers_by_status = (
    orders.groupby("order_status")
    .agg(
        approval_outliers=(
            "approval_time_outlier",
            "sum"
        ),
        delivery_outliers=(
            "delivery_time_outlier",
            "sum"
        )
    )
    .sort_values(
        "delivery_outliers",
        ascending=False
    )
)

outliers_by_status

99th-percentile approval threshold: 90.17 hours
99th-percentile delivery threshold: 46.05 days
Approval-time outliers: 993
Delivery-time outliers: 965


,approval_outliers,delivery_outliers
order_status,,
delivered,940,965
approved,1,0
canceled,9,0
created,0,0
invoiced,3,0
processing,3,0
shipped,12,0
unavailable,25,0


## 6. Validate Transactional Measures

Prices, freight charges, payment values, installment counts, review scores, and physical product measurements are checked for impossible or invalid values before aggregation.

In [11]:
numeric_validation_checks = [
    {
        "check": "Negative item prices",
        "invalid_rows": int(
            (order_items["price"] < 0).sum()
        )
    },
    {
        "check": "Negative freight values",
        "invalid_rows": int(
            (order_items["freight_value"] < 0).sum()
        )
    },
    {
        "check": "Negative payment values",
        "invalid_rows": int(
            (payments["payment_value"] < 0).sum()
        )
    },
    {
        "check": "Installment counts below 1",
        "invalid_rows": int(
            (payments["payment_installments"] < 1).sum()
        )
    },
    {
        "check": "Review scores outside 1–5",
        "invalid_rows": int(
            (~reviews["review_score"].between(1, 5)).sum()
        )
    },
    {
        "check": "Nonpositive product weights",
        "invalid_rows": int(
            (
                products["product_weight_g"].notna()
                & (products["product_weight_g"] <= 0)
            ).sum()
        )
    },
    {
        "check": "Nonpositive product lengths",
        "invalid_rows": int(
            (
                products["product_length_cm"].notna()
                & (products["product_length_cm"] <= 0)
            ).sum()
        )
    },
    {
        "check": "Nonpositive product heights",
        "invalid_rows": int(
            (
                products["product_height_cm"].notna()
                & (products["product_height_cm"] <= 0)
            ).sum()
        )
    },
    {
        "check": "Nonpositive product widths",
        "invalid_rows": int(
            (
                products["product_width_cm"].notna()
                & (products["product_width_cm"] <= 0)
            ).sum()
        )
    }
]

numeric_validation = pd.DataFrame(
    numeric_validation_checks
)

numeric_validation

,check,invalid_rows
0,Negative item prices,0
1,Negative freight values,0
2,Negative payment values,0
3,Installment counts below 1,2
4,Review scores outside 1–5,0
5,Nonpositive product weights,4
6,Nonpositive product lengths,0
7,Nonpositive product heights,0
8,Nonpositive product widths,0


In [12]:
invalid_installments = payments[
    payments["payment_installments"] < 1
].copy()

invalid_installments

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [13]:
invalid_product_weights = products[
    products["product_weight_g"].notna()
    & (products["product_weight_g"] <= 0)
][
    [
        "product_id",
        "product_category_name",
        "product_category",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
].copy()

invalid_product_weights

,product_id,product_category_name,product_category,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,bed_bath_table,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,bed_bath_table,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,bed_bath_table,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,bed_bath_table,0.0,30.0,25.0,30.0


### Numeric Data Corrections

Two positive credit-card payments recorded zero installments. These records are flagged and corrected to one installment, representing a single payment.

Four products recorded impossible zero weights. These records are retained and flagged, while their weights are converted to missing values rather than estimated.

In [14]:
payments["installment_count_corrected"] = (
    payments["payment_installments"] < 1
)

payments.loc[
    payments["installment_count_corrected"],
    "payment_installments"
] = 1

products["product_weight_invalid"] = (
    products["product_weight_g"].notna()
    & (products["product_weight_g"] <= 0)
)

products.loc[
    products["product_weight_invalid"],
    "product_weight_g"
] = np.nan

assert (payments["payment_installments"] < 1).sum() == 0

assert (
    products["product_weight_g"].notna()
    & (products["product_weight_g"] <= 0)
).sum() == 0

print(
    "Payment installment corrections:",
    f"{payments['installment_count_corrected'].sum():,}"
)

print(
    "Invalid product weights converted to missing:",
    f"{products['product_weight_invalid'].sum():,}"
)

Payment installment corrections: 2
Invalid product weights converted to missing: 4


## 7. Aggregate Order-Item Metrics

Individual product lines are aggregated into one record per order. This protects financial measures from duplication during later joins.

In [15]:
order_items["item_total_value"] = (
    order_items["price"]
    + order_items["freight_value"]
)

order_item_summary = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "count"),
        unique_product_count=("product_id", "nunique"),
        unique_seller_count=("seller_id", "nunique"),
        product_value=("price", "sum"),
        freight_value=("freight_value", "sum"),
        total_order_value=("item_total_value", "sum"),
        average_item_price=("price", "mean")
    )
)

order_item_summary["freight_share"] = (
    order_item_summary["freight_value"]
    / order_item_summary["total_order_value"]
)

assert order_item_summary["order_id"].is_unique

assert np.isclose(
    order_item_summary["product_value"].sum(),
    order_items["price"].sum()
)

assert np.isclose(
    order_item_summary["freight_value"].sum(),
    order_items["freight_value"].sum()
)

print(
    f"Original item rows: "
    f"{len(order_items):,}"
)

print(
    f"Orders containing item data: "
    f"{len(order_item_summary):,}"
)

print(
    f"Total product value: "
    f"{order_item_summary['product_value'].sum():,.2f}"
)

print(
    f"Total freight value: "
    f"{order_item_summary['freight_value'].sum():,.2f}"
)

print(
    "Orders with multiple products:",
    f"{(order_item_summary['unique_product_count'] > 1).sum():,}"
)

print(
    "Orders with multiple sellers:",
    f"{(order_item_summary['unique_seller_count'] > 1).sum():,}"
)

order_item_summary.head()

Original item rows: 112,650
Orders containing item data: 98,666
Total product value: 13,591,643.70
Total freight value: 2,251,909.54
Orders with multiple products: 3,236
Orders with multiple sellers: 1,278


,order_id,item_count,unique_product_count,unique_seller_count,product_value,freight_value,total_order_value,average_item_price,freight_share
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29,72.19,58.90,0.184098
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93,259.83,239.90,0.076704
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87,216.87,199.00,0.082400
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79,25.78,12.99,0.496121
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14,218.04,199.90,0.083196


## 8. Aggregate and Reconcile Payments

Payment records are aggregated to the order level. Recorded payment totals are compared with product-plus-freight totals to identify discrepancies without double-counting multi-method payments.

In [16]:
payment_summary = (
    payments
    .groupby("order_id", as_index=False)
    .agg(
        payment_record_count=(
            "payment_sequential",
            "count"
        ),
        payment_method_count=(
            "payment_type",
            "nunique"
        ),
        payment_methods=(
            "payment_type",
            lambda values: ", ".join(
                sorted(set(values))
            )
        ),
        maximum_installments=(
            "payment_installments",
            "max"
        ),
        total_payment_value=(
            "payment_value",
            "sum"
        )
    )
)

primary_payment_method = (
    payments
    .sort_values(
        ["order_id", "payment_value"],
        ascending=[True, False]
    )
    .drop_duplicates("order_id")
    [["order_id", "payment_type"]]
    .rename(
        columns={
            "payment_type": "primary_payment_method"
        }
    )
)

payment_summary = payment_summary.merge(
    primary_payment_method,
    how="left",
    on="order_id",
    validate="one_to_one"
)

assert payment_summary["order_id"].is_unique

financial_reconciliation = order_item_summary.merge(
    payment_summary,
    how="outer",
    on="order_id",
    validate="one_to_one"
)

both_values_available = (
    financial_reconciliation["total_order_value"].notna()
    & financial_reconciliation["total_payment_value"].notna()
)

financial_reconciliation["payment_difference"] = (
    financial_reconciliation["total_payment_value"]
    - financial_reconciliation["total_order_value"]
)

financial_reconciliation["payment_reconciled"] = pd.Series(
    pd.NA,
    index=financial_reconciliation.index,
    dtype="boolean"
)

financial_reconciliation.loc[
    both_values_available,
    "payment_reconciled"
] = (
    financial_reconciliation.loc[
        both_values_available,
        "payment_difference"
    ].abs() <= 0.01
)

print(f"Original payment rows: {len(payments):,}")
print(
    f"Orders containing payment data: "
    f"{len(payment_summary):,}"
)
print(
    "Orders using multiple payment methods:",
    f"{(payment_summary['payment_method_count'] > 1).sum():,}"
)
print(
    "Orders successfully reconciled:",
    f"{financial_reconciliation['payment_reconciled'].sum():,}"
)
print(
    "Orders with payment discrepancies:",
    f"{(financial_reconciliation['payment_reconciled'] == False).sum():,}"
)
print(
    "Orders missing item or payment information:",
    f"{(~both_values_available).sum():,}"
)

Original payment rows: 103,886
Orders containing payment data: 99,440
Orders using multiple payment methods: 2,246
Orders successfully reconciled: 98,287
Orders with payment discrepancies: 378
Orders missing item or payment information: 776


### Financial Reconciliation Investigation

Payment totals are compared with product-plus-freight totals after rounding to currency precision. Discrepancies and missing financial records are analyzed by order status before any exclusion decisions are made.

In [17]:
reconciliation_detail = financial_reconciliation.merge(
    orders[["order_id", "order_status"]],
    how="left",
    on="order_id",
    validate="one_to_one"
)

reconciliation_detail["item_data_missing"] = (
    reconciliation_detail["total_order_value"].isna()
)

reconciliation_detail["payment_data_missing"] = (
    reconciliation_detail["total_payment_value"].isna()
)

reconciliation_detail["has_both_financial_values"] = (
    ~reconciliation_detail["item_data_missing"]
    & ~reconciliation_detail["payment_data_missing"]
)

reconciliation_detail["payment_difference"] = (
    reconciliation_detail["total_payment_value"]
    - reconciliation_detail["total_order_value"]
).round(2)

reconciliation_detail["absolute_payment_difference"] = (
    reconciliation_detail["payment_difference"].abs()
)

reconciliation_detail["payment_discrepancy"] = (
    reconciliation_detail["has_both_financial_values"]
    & (
        reconciliation_detail[
            "absolute_payment_difference"
        ] > 0.01
    )
)

reconciliation_by_status = (
    reconciliation_detail
    .groupby("order_status", dropna=False)
    .agg(
        order_count=("order_id", "count"),
        missing_item_data=("item_data_missing", "sum"),
        missing_payment_data=("payment_data_missing", "sum"),
        payment_discrepancies=("payment_discrepancy", "sum"),
        maximum_absolute_difference=(
            "absolute_payment_difference",
            "max"
        )
    )
    .sort_values("order_count", ascending=False)
)

reconciliation_by_status

,order_count,missing_item_data,missing_payment_data,payment_discrepancies,maximum_absolute_difference
order_status,,,,,
delivered,96478,0,1,299,182.81
shipped,1107,1,0,2,5.99
canceled,625,164,0,2,25.12
unavailable,609,603,0,0,0.00
invoiced,314,2,0,0,0.01
processing,301,0,0,0,0.00
created,5,5,0,0,NaN
approved,2,0,0,0,0.00


In [18]:
largest_payment_discrepancies = (
    reconciliation_detail[
        reconciliation_detail["payment_discrepancy"]
    ]
    [
        [
            "order_id",
            "order_status",
            "total_order_value",
            "total_payment_value",
            "payment_difference"
        ]
    ]
    .sort_values(
        "payment_difference",
        key=abs,
        ascending=False
    )
    .head(10)
)

largest_payment_discrepancies

,order_id,order_status,total_order_value,total_payment_value,payment_difference
80176,ce6d150fb29ada17d2082f4847107665,delivered,1403.66,1586.47,182.81
42865,6e5fe7366a2e1bfbf3257dba0af1267f,delivered,287.91,406.92,119.01
43791,70b742795bc441e94a44a084b6d9ce7a,delivered,466.93,578.82,111.89
59228,996c7e73600ad3723e8627ab7bef81e4,delivered,587.90,664.43,76.53
43794,70b7e94ea46d3e8b5bc12a50186edaf0,delivered,213.15,274.84,61.69
73090,bc2c82b0ef78d2252b6176d1972db7c9,delivered,242.01,303.02,61.01
68015,af9ffff2ce6b3defd34fd4c78857a379,delivered,413.17,466.97,53.80
14745,262118ce178bb3e4590a3adcf6d62e6b,delivered,177.74,126.12,-51.62
74503,bfdb5bbb06458d600a33d61f5f287472,delivered,348.93,394.36,45.43
54744,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,delivered,254.45,293.89,39.44


## 9. Aggregate Customer Reviews

Review records are aggregated to one row per order. Review counts and average scores preserve the full history, while the most recently answered review represents the latest customer assessment.

In [19]:
reviews["has_written_comment"] = (
    reviews["review_comment_message"]
    .fillna("")
    .str.strip()
    .ne("")
)

review_summary = (
    reviews
    .groupby("order_id", as_index=False)
    .agg(
        review_count=("review_id", "count"),
        average_review_score=("review_score", "mean"),
        written_review_count=(
            "has_written_comment",
            "sum"
        ),
        first_review_date=(
            "review_creation_date",
            "min"
        ),
        latest_review_answer_timestamp=(
            "review_answer_timestamp",
            "max"
        )
    )
)

latest_review = (
    reviews
    .sort_values(
        [
            "order_id",
            "review_answer_timestamp",
            "review_creation_date"
        ]
    )
    .drop_duplicates(
        subset=["order_id"],
        keep="last"
    )
    [
        [
            "order_id",
            "review_score",
            "review_comment_title",
            "review_comment_message"
        ]
    ]
    .rename(
        columns={
            "review_score": "latest_review_score",
            "review_comment_title":
                "latest_review_title",
            "review_comment_message":
                "latest_review_message"
        }
    )
)

review_summary = review_summary.merge(
    latest_review,
    how="left",
    on="order_id",
    validate="one_to_one"
)

assert review_summary["order_id"].is_unique

print(f"Original review rows: {len(reviews):,}")
print(
    f"Orders containing reviews: "
    f"{len(review_summary):,}"
)
print(
    "Orders with multiple reviews:",
    f"{(review_summary['review_count'] > 1).sum():,}"
)
print(
    "Orders with written comments:",
    f"{(review_summary['written_review_count'] > 0).sum():,}"
)
print(
    f"Overall average review score: "
    f"{reviews['review_score'].mean():.2f}"
)

review_summary.head()

Original review rows: 99,224
Orders containing reviews: 98,673
Orders with multiple reviews: 547
Orders with written comments: 40,809
Overall average review score: 4.09


,order_id,review_count,average_review_score,written_review_count,first_review_date,latest_review_answer_timestamp,latest_review_score,latest_review_title,latest_review_message
0,00010242fe8c5a6d1ba2dd792cb16214,1,5.0,1,2017-09-21,2017-09-22 10:57:03,5,NaN,"Perfeito, produto entregue antes do combinado."
1,00018f77f2f0320c557190d7a144bdd3,1,4.0,0,2017-05-13,2017-05-15 11:34:13,4,NaN,NaN
2,000229ec398224ef6ca0657da4fc703e,1,5.0,1,2018-01-23,2018-01-23 16:06:31,5,NaN,Chegou antes do prazo previsto e o produto sur...
3,00024acbcdf0a6daa1e931b038114c75,1,4.0,0,2018-08-15,2018-08-15 16:39:01,4,NaN,NaN
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,5.0,1,2017-03-02,2017-03-03 10:54:59,5,NaN,Gostei pois veio no prazo determinado .


## 10. Build the Order-Level Analytics Table

Cleaned customer, item, payment, delivery, and review data are combined into a single table with exactly one record per order. One-to-one and many-to-one merge validation prevents accidental duplication.

In [20]:
order_fact = (
    orders
    .merge(
        customers,
        how="left",
        on="customer_id",
        validate="many_to_one"
    )
    .merge(
        order_item_summary,
        how="left",
        on="order_id",
        validate="one_to_one"
    )
    .merge(
        payment_summary,
        how="left",
        on="order_id",
        validate="one_to_one"
    )
    .merge(
        review_summary,
        how="left",
        on="order_id",
        validate="one_to_one"
    )
)

order_fact["has_item_data"] = (
    order_fact["total_order_value"].notna()
)

order_fact["has_payment_data"] = (
    order_fact["total_payment_value"].notna()
)

order_fact["has_review_data"] = (
    order_fact["latest_review_score"].notna()
)

order_fact["is_completed_order"] = (
    order_fact["order_status"] == "delivered"
)

order_fact["payment_difference"] = (
    order_fact["total_payment_value"]
    - order_fact["total_order_value"]
).round(2)

financial_values_available = (
    order_fact["has_item_data"]
    & order_fact["has_payment_data"]
)

order_fact["payment_reconciled"] = pd.Series(
    pd.NA,
    index=order_fact.index,
    dtype="boolean"
)

order_fact.loc[
    financial_values_available,
    "payment_reconciled"
] = (
    order_fact.loc[
        financial_values_available,
        "payment_difference"
    ].abs() <= 0.01
)

count_columns = [
    "item_count",
    "unique_product_count",
    "unique_seller_count",
    "payment_record_count",
    "payment_method_count",
    "review_count",
    "written_review_count"
]

for column in count_columns:
    order_fact[column] = (
        order_fact[column]
        .fillna(0)
        .astype("Int64")
    )

order_fact = order_fact.sort_values(
    "order_purchase_timestamp"
)

order_fact["customer_order_number"] = (
    order_fact
    .groupby("customer_unique_id")
    .cumcount()
    + 1
)

order_fact["is_repeat_customer_order"] = (
    order_fact["customer_order_number"] > 1
)

order_fact = order_fact.reset_index(drop=True)

assert len(order_fact) == len(orders)
assert order_fact["order_id"].is_unique
assert order_fact["customer_id"].notna().all()

print(f"Master order rows:       {len(order_fact):,}")
print(f"Master order columns:    {len(order_fact.columns):,}")
print(
    f"Delivered orders:        "
    f"{order_fact['is_completed_order'].sum():,}"
)
print(
    f"Orders with item data:   "
    f"{order_fact['has_item_data'].sum():,}"
)
print(
    f"Orders with payments:    "
    f"{order_fact['has_payment_data'].sum():,}"
)
print(
    f"Orders with reviews:     "
    f"{order_fact['has_review_data'].sum():,}"
)
print(
    f"Repeat-customer orders:  "
    f"{order_fact['is_repeat_customer_order'].sum():,}"
)

Master order rows:       99,441
Master order columns:    55
Delivered orders:        96,478
Orders with item data:   98,666
Orders with payments:    99,440
Orders with reviews:     98,673
Repeat-customer orders:  3,345


## 11. Build the Item-Level Product Performance Table

Each purchased item is enriched with product category, seller location, customer location, delivery performance, and customer-review outcomes. Financial fields remain at the item level to prevent double-counting.

In [22]:
order_context_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id",
    "order_status",
    "order_purchase_timestamp",
    "purchase_date",
    "purchase_month",
    "actual_delivery_days",
    "days_from_estimate",
    "is_late_delivery",
    "latest_review_score",
    "customer_city",
    "customer_state",
    "customer_latitude",
    "customer_longitude"
]

order_context = order_fact[order_context_columns].copy()

item_fact = (
    order_items
    .merge(
        products,
        how="left",
        on="product_id",
        validate="many_to_one"
    )
    .merge(
        sellers,
        how="left",
        on="seller_id",
        validate="many_to_one"
    )
    .merge(
        order_context,
        how="left",
        on="order_id",
        validate="many_to_one"
    )
)

item_fact["freight_percentage"] = np.where(
    item_fact["item_total_value"] > 0,
    (
        item_fact["freight_value"]
        / item_fact["item_total_value"]
    ) * 100,
    np.nan
)

coordinate_data_available = (
    item_fact["customer_latitude"].notna()
    & item_fact["customer_longitude"].notna()
    & item_fact["seller_latitude"].notna()
    & item_fact["seller_longitude"].notna()
)

customer_latitude = np.radians(
    item_fact["customer_latitude"]
)

customer_longitude = np.radians(
    item_fact["customer_longitude"]
)

seller_latitude = np.radians(
    item_fact["seller_latitude"]
)

seller_longitude = np.radians(
    item_fact["seller_longitude"]
)

latitude_difference = (
    customer_latitude - seller_latitude
)

longitude_difference = (
    customer_longitude - seller_longitude
)

haversine_value = (
    np.sin(latitude_difference / 2) ** 2
    + np.cos(seller_latitude)
    * np.cos(customer_latitude)
    * np.sin(longitude_difference / 2) ** 2
)

item_fact["seller_customer_distance_km"] = np.where(
    coordinate_data_available,
    2 * 6371 * np.arcsin(
        np.sqrt(haversine_value)
    ),
    np.nan
)

assert len(item_fact) == len(order_items)

assert not item_fact[
    ["order_id", "order_item_id"]
].duplicated().any()

assert np.isclose(
    item_fact["price"].sum(),
    order_items["price"].sum()
)

print(f"Item-level rows:          {len(item_fact):,}")
print(f"Item-level columns:       {len(item_fact.columns):,}")
print(
    "Missing product categories:",
    f"{item_fact['product_category'].isna().sum():,}"
)
print(
    "Location-distance coverage:",
    f"{item_fact['seller_customer_distance_km'].notna().mean() * 100:.2f}%"
)
print(
    "Total product value preserved:",
    f"{item_fact['price'].sum():,.2f}"
)
print(
    "Total freight value preserved:",
    f"{item_fact['freight_value'].sum():,.2f}"
)

Item-level rows:          112,650
Item-level columns:       41
Missing product categories: 0
Location-distance coverage: 99.51%
Total product value preserved: 13,591,643.70
Total freight value preserved: 2,251,909.54


## 12. Export the Analytics-Ready Datasets

The cleaned relational tables and two denormalized analytics tables are exported for SQL analysis, dashboard development, and reproducible downstream use.

In [23]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

export_tables = {
    # Clean relational tables
    "customers_clean.csv": customers,
    "sellers_clean.csv": sellers,
    "products_clean.csv": products,
    "orders_clean.csv": orders,
    "order_items_clean.csv": order_items,
    "payments_clean.csv": payments,
    "reviews_clean.csv": reviews,
    "geolocation_clean.csv": geolocation_clean,

    # Analytics-ready tables
    "order_fact.csv": order_fact,
    "item_fact.csv": item_fact,
    "payment_summary.csv": payment_summary,
    "review_summary.csv": review_summary,
}

export_results = []

for filename, dataframe in export_tables.items():
    output_path = PROCESSED_DATA_DIR / filename
    dataframe.to_csv(output_path, index=False)

    export_results.append({
        "file": filename,
        "rows": len(dataframe),
        "columns": len(dataframe.columns),
        "size_mb": round(output_path.stat().st_size / 1_000_000, 2),
    })

export_summary = pd.DataFrame(export_results)

assert len(order_fact) == 99_441
assert len(item_fact) == 112_650
assert all(
    (PROCESSED_DATA_DIR / filename).exists()
    for filename in export_tables
)

print(f"Exported {len(export_tables)} processed datasets.")
export_summary

Exported 12 processed datasets.


,file,rows,columns,size_mb
0,customers_clean.csv,99441,8,12.88
1,sellers_clean.csv,3095,7,0.30
2,products_clean.csv,32951,12,3.65
3,orders_clean.csv,99441,18,28.23
4,order_items_clean.csv,112650,8,15.89
5,payments_clean.csv,103886,6,6.26
6,reviews_clean.csv,99224,8,14.51
7,geolocation_clean.csv,19011,3,0.84
8,order_fact.csv,99441,55,57.83
9,item_fact.csv,112650,41,60.34


## 13. Final Data-Quality Validation

A final quality gate verifies table grain, key integrity, referential integrity, financial preservation, corrected business rules, category completeness, and geographic coverage before the datasets are loaded into SQL.

In [24]:
validation_records = []

def add_validation(check, passed, observed, expected):
    validation_records.append({
        "check": check,
        "passed": bool(passed),
        "observed": observed,
        "expected": expected,
    })

# Table grain and primary keys
add_validation(
    "Order fact row count",
    len(order_fact) == len(orders),
    len(order_fact),
    len(orders),
)

add_validation(
    "Order fact has one row per order",
    order_fact["order_id"].is_unique,
    order_fact["order_id"].nunique(),
    len(order_fact),
)

add_validation(
    "Item fact row count",
    len(item_fact) == len(order_items),
    len(item_fact),
    len(order_items),
)

add_validation(
    "Item fact compound key is unique",
    not item_fact[["order_id", "order_item_id"]].duplicated().any(),
    int(item_fact[["order_id", "order_item_id"]].duplicated().sum()),
    0,
)

# Referential integrity
add_validation(
    "All item orders exist in order fact",
    item_fact["order_id"].isin(order_fact["order_id"]).all(),
    int((~item_fact["order_id"].isin(order_fact["order_id"])).sum()),
    0,
)

add_validation(
    "All item products exist in product dimension",
    item_fact["product_id"].isin(products["product_id"]).all(),
    int((~item_fact["product_id"].isin(products["product_id"])).sum()),
    0,
)

add_validation(
    "All item sellers exist in seller dimension",
    item_fact["seller_id"].isin(sellers["seller_id"]).all(),
    int((~item_fact["seller_id"].isin(sellers["seller_id"])).sum()),
    0,
)

# Financial preservation
add_validation(
    "Product value preserved",
    np.isclose(
        item_fact["price"].sum(),
        order_items["price"].sum(),
        atol=0.01,
    ),
    round(item_fact["price"].sum(), 2),
    round(order_items["price"].sum(), 2),
)

add_validation(
    "Freight value preserved",
    np.isclose(
        item_fact["freight_value"].sum(),
        order_items["freight_value"].sum(),
        atol=0.01,
    ),
    round(item_fact["freight_value"].sum(), 2),
    round(order_items["freight_value"].sum(), 2),
)

# Corrected business rules
add_validation(
    "Installment counts are at least one",
    (payments["payment_installments"] >= 1).all(),
    int((payments["payment_installments"] < 1).sum()),
    0,
)

add_validation(
    "Product weights are positive or missing",
    (products["product_weight_g"].dropna() > 0).all(),
    int((products["product_weight_g"].dropna() <= 0).sum()),
    0,
)

add_validation(
    "No negative item prices",
    (order_items["price"] >= 0).all(),
    int((order_items["price"] < 0).sum()),
    0,
)

add_validation(
    "No negative payment values",
    (payments["payment_value"] >= 0).all(),
    int((payments["payment_value"] < 0).sum()),
    0,
)

# Analytical completeness
add_validation(
    "Every item has a product category",
    item_fact["product_category"].notna().all(),
    int(item_fact["product_category"].isna().sum()),
    0,
)

distance_coverage = item_fact["seller_customer_distance_km"].notna().mean()

add_validation(
    "Location-distance coverage exceeds 99%",
    distance_coverage >= 0.99,
    f"{distance_coverage:.2%}",
    "At least 99%",
)

add_validation(
    "No order timeline anomalies",
    int(orders["timeline_anomaly"].sum()) == 0,
    int(orders["timeline_anomaly"].sum()),
    0,
)

validation_summary = pd.DataFrame(validation_records)

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

validation_summary.to_csv(
    REPORTS_DIR / "final_validation_report.csv",
    index=False,
)

passed_checks = int(validation_summary["passed"].sum())
total_checks = len(validation_summary)

print(f"Final validation: {passed_checks}/{total_checks} checks passed.")
validation_summary

Final validation: 16/16 checks passed.


,check,passed,observed,expected
0,Order fact row count,True,99441,99441
1,Order fact has one row per order,True,99441,99441
2,Item fact row count,True,112650,112650
3,Item fact compound key is unique,True,0,0
4,All item orders exist in order fact,True,0,0
5,All item products exist in product dimension,True,0,0
6,All item sellers exist in seller dimension,True,0,0
7,Product value preserved,True,13591643.7,13591643.7
8,Freight value preserved,True,2251909.54,2251909.54
9,Installment counts are at least one,True,0,0
